# 01 - Build Benchmark

This notebook generates and validates the Backtest Lie Detector benchmark dataset.

## Contents
1. Load and explore seed cases
2. Generate additional cases
3. Validate ground truth
4. Export benchmark files

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from pathlib import Path

from backtest_lie_detector.schemas import Module, Difficulty, Validity, ViolationType
from backtest_lie_detector.benchmark.examples import SEED_CASES
from backtest_lie_detector.benchmark.build_cases import (
    generate_all_cases,
    save_benchmark,
    load_benchmark,
    print_benchmark_summary,
)
from backtest_lie_detector.benchmark.validators import validate_all_cases

## 1. Explore Seed Cases

In [ ]:
print(f"Number of seed cases: {len(SEED_CASES)}")

# Show first case as example
case = SEED_CASES[0]
print(f"\nExample case:")
print(f"  ID: {case.id}")
print(f"  Module: {case.module.value}")
print(f"  Difficulty: {case.difficulty.value}")
print(f"  Prompt: {case.prompt[:100]}...")
print(f"  Expected validity: {case.expected_validity.value}")
print(f"  Expected violations: {[v.value for v in case.expected_violations]}")

## 2. Generate All Cases

In [ ]:
# Generate the full benchmark
all_cases = generate_all_cases()
print_benchmark_summary(all_cases)

In [ ]:
# Convert to DataFrame for easier exploration
cases_df = pd.DataFrame([{
    'id': c.id,
    'module': c.module.value,
    'difficulty': c.difficulty.value,
    'expected_validity': c.expected_validity.value,
    'violations': ','.join(v.value for v in c.expected_violations),
    'n_violations': len(c.expected_violations),
    'prompt_length': len(c.prompt),
} for c in all_cases])

cases_df.head(10)

In [ ]:
# Distribution by module
cases_df.groupby('module').size()

In [ ]:
# Distribution by difficulty
cases_df.groupby('difficulty').size()

In [ ]:
# Distribution by expected validity
cases_df.groupby('expected_validity').size()

## 3. Validate Ground Truth

In [ ]:
# Run validators on all cases
validations = validate_all_cases(all_cases)

# Count consistent vs inconsistent
consistent = [v for v in validations if v.get('consistent') == True]
inconsistent = [v for v in validations if v.get('consistent') == False]
no_validation = [v for v in validations if v.get('consistent') is None]

print(f"Validated as consistent: {len(consistent)}")
print(f"Validated as inconsistent: {len(inconsistent)}")
print(f"No automated validation available: {len(no_validation)}")

In [ ]:
# Show any inconsistencies
if inconsistent:
    print("\nInconsistent cases (may need manual review):")
    for v in inconsistent:
        print(f"  - {v['case_id']}: {v['validations']}")
else:
    print("\nNo inconsistencies detected!")

## 4. Export Benchmark

In [ ]:
# Save full benchmark
output_path = Path('../data/benchmark/benchmark_v1.jsonl')
save_benchmark(all_cases, str(output_path))

# Save sample for quick testing
sample_path = Path('../data/benchmark/benchmark_sample.jsonl')
save_benchmark(all_cases[:20], str(sample_path))

In [ ]:
# Verify we can reload
reloaded = load_benchmark(str(output_path))
print(f"Reloaded {len(reloaded)} cases successfully")

## Next Steps

1. Review the generated cases manually for quality
2. Add more cases if needed (target: 80+ cases)
3. Run `02_run_evals.ipynb` to evaluate models
4. Run `03_analyze_results.ipynb` to analyze results